In [ ]:
import json
import csv
from pathlib import Path
from collections import defaultdict
from sentence_transformers import SentenceTransformer
import numpy as np


# Install dependencies (run once)# !pip install sentence-transformers

In [ ]:
# The two models to compare
BASE_MODEL = "gemma2:9b"        # Baseline explanations (ground truth)
COMPARE_MODEL = "llama3.1:8b"    # Explanations to compare

BASE_TAG = BASE_MODEL.replace(":" , "_").replace(".", "_").replace("-", "_")
COMPARE_TAG = COMPARE_MODEL.replace(":", "_").replace(".", "_").replace("-", "_")

print(f"[INFO] Base model (baseline explanations): {BASE_MODEL} -> {BASE_TAG}")
print(f"[INFO] Compare model: {COMPARE_MODEL} -> {COMPARE_TAG}")

[INFO] Base model (ground truth): deepseek-r1:14b -> deepseek_r1_14b
[INFO] Compare model: llama3.1:8b -> llama3_1_8b


In [ ]:
SA_DATASET_NAMES = ["DoS", "Fuzzy", "Gear", "RPM"]

SA_QUESTION_FILES = {
    "DoS":   Path("DoS_sa_qa/questions/dos_sa_questions.json"),
    "Fuzzy": Path("Fuzzy_sa_qa/questions/fuzzy_sa_questions.json"),
    "Gear":  Path("Gear_sa_qa/questions/gear_sa_questions.json"),
    "RPM":   Path("RPM_sa_qa/questions/rpm_sa_questions.json"),
}

def sa_answer_path(dataset_name: str, model_tag: str) -> Path:
    """
    Get the path to answer/explanation file for a given model.
    Checks for explanation files first (gemma2), then falls back to answer files.
    """
    # Check for explanation file (gemma2 generates these)
    explanation_path = Path(f"{dataset_name}_sa_qa/llm_answers/{dataset_name.lower()}_sa_explanations_{model_tag}.jsonl")
    if explanation_path.exists():
        return explanation_path
    
    # Fall back to regular answer file (other models)
    return Path(f"{dataset_name}_sa_qa/llm_answers/{dataset_name.lower()}_sa_answers_{model_tag}.jsonl")

In [5]:
def load_sa_answers(path: Path) -> list:
    """Load answer records from JSONL file."""
    records = []
    if not path.exists():
        return records
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

def normalize_text(text: str) -> str:
    return " ".join(text.strip().split()).lower() if text else ""

In [ ]:
# Load semantic similarity model
print("[INFO] Loading sentence transformer model...")
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
print("[INFO] Model loaded successfully")


def compute_token_f1(pred: str, gold: str) -> float:
    """Token-level F1 score: overlap of tokens between prediction and gold."""
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0
    
    pred_tokens = set(pred.split())
    gold_tokens = set(gold.split())
    
    if not pred_tokens or not gold_tokens:
        return 0.0
    
    intersection = len(pred_tokens & gold_tokens)
    precision = intersection / len(pred_tokens)
    recall = intersection / len(gold_tokens)
    
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)


def compute_rouge_l(pred: str, gold: str) -> float:
    """ROUGE-L: Longest Common Subsequence based F1."""
    if not pred and not gold:
        return 1.0
    if not pred or not gold:
        return 0.0
    
    pred_tokens = pred.split()
    gold_tokens = gold.split()
    
    # Longest Common Subsequence
    m, n = len(pred_tokens), len(gold_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred_tokens[i-1] == gold_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    
    lcs_len = dp[m][n]

    precision = lcs_len / len(pred_tokens) if pred_tokens else 0.0        return 0.0

    recall = lcs_len / len(gold_tokens) if gold_tokens else 0.0        print(f"[WARN] Semantic similarity error: {e}")

        except Exception as e:

    if precision + recall == 0:        return float(similarity)

        return 0.0        )

    return 2 * (precision * recall) / (precision + recall)            np.linalg.norm(embeddings[0]) * np.linalg.norm(embeddings[1])

        similarity = np.dot(embeddings[0], embeddings[1]) / (

        # Compute cosine similarity

def compute_semantic_similarity(pred: str, gold: str) -> float:        embeddings = semantic_model.encode([pred, gold], convert_to_numpy=True)

    """Fast semantic similarity using sentence-transformers."""        # Encode both texts

    if not pred and not gold:    try:

        return 1.0    

    if not pred or not gold:        return 0.0

In [ ]:
def compute_comparison_records(base_answers: list, compare_answers: list) -> list:
    """
    Compare explanations from two models.
    base_answers: baseline explanations (e.g., gemma2:9b)
    compare_answers: explanations to compare (e.g., llama3.1:8b)
    """
    # Create a map of base answers by qa_id
    base_map = {rec.get("qa_id"): rec for rec in base_answers}
    
    eval_records = []
    
    for rec in compare_answers:
        qa_id = rec.get("qa_id")
        if qa_id is None or qa_id not in base_map:
            continue
        
        base_rec = base_map[qa_id]
        
        # Get the base model explanation (this is our baseline for comparison)
        gold_raw = base_rec.get("llm_answer", "")
        # Get the compare model explanation
        pred_raw = rec.get("llm_answer", "")
        
        pred = normalize_text(pred_raw)
        gold = normalize_text(gold_raw)
        
        # Compute metrics
        exact_match = 1.0 if pred == gold else 0.0
        token_f1 = compute_token_f1(pred, gold)
        rouge_l = compute_rouge_l(pred, gold)
        semantic_sim = compute_semantic_similarity(pred_raw, gold_raw)
        
        eval_records.append({
            "qa_id": qa_id,
            "sa_type": rec.get("sa_type", "unknown"),
            "base_answer": gold_raw,
            "compare_answer": pred_raw,
            "base_normalized": gold,
            "compare_normalized": pred,
            "exact_match": exact_match,
            "token_f1": token_f1,
            "rouge_l": rouge_l,
            "semantic_similarity": semantic_sim,
        })
    
    return eval_records


def accuracy_from_records(records: list) -> dict:
    """Calculate Token F1, ROUGE-L, and Semantic Similarity from records."""
    if not records:
        return {"token_f1": 0.0, "rouge_l": 0.0, "semantic_similarity": 0.0, "total": 0}
    
    total = len(records)
    token_f1 = sum(r["token_f1"] for r in records) / total if total else 0.0
    rouge_l = sum(r["rouge_l"] for r in records) / total if total else 0.0
    semantic_similarity = sum(r["semantic_similarity"] for r in records) / total if total else 0.0
    
    return {
        "token_f1": token_f1,
        "rouge_l": rouge_l,
        "semantic_similarity": semantic_similarity,
        "total": total
    }


def metrics_by_category(records: list) -> dict:
    """Group records by sa_type and compute metrics for each category."""
    from collections import defaultdict
    
    category_records = defaultdict(list)
    for rec in records:
        sa_type = rec.get("sa_type", "unknown")
        category_records[sa_type].append(rec)
    
    category_metrics = {}
    for sa_type, cat_records in category_records.items():
        category_metrics[sa_type] = accuracy_from_records(cat_records)
    
    return category_metrics

In [ ]:
OUTPUT_DIR = Path("LLM_SA_Comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_csv = OUTPUT_DIR / f"Comparison_{BASE_TAG}_vs_{COMPARE_TAG}.csv"

print(f"[DEBUG] Current working directory: {Path.cwd()}")
print(f"[DEBUG] Base model: {BASE_MODEL} ({BASE_TAG})")
print(f"[DEBUG] Compare model: {COMPARE_MODEL} ({COMPARE_TAG})")
print(f"[DEBUG] Output CSV: {out_csv}")

header = ["Attack", "Total", "Token F1", "ROUGE-L", "Semantic Similarity"]
all_records = []

with out_csv.open("w", encoding="utf-8", newline="") as f_out:
    writer = csv.writer(f_out)
    writer.writerow(header)

    for name in SA_DATASET_NAMES:
        base_path = sa_answer_path(name, BASE_TAG)
        compare_path = sa_answer_path(name, COMPARE_TAG)
        
        print(f"[DEBUG] Processing {name}:")
        print(f"  Base answers path: {base_path} -> exists: {base_path.exists()}")
        print(f"  Compare answers path: {compare_path} -> exists: {compare_path.exists()}")

        base_records = load_sa_answers(base_path)
        compare_records = load_sa_answers(compare_path)
        
        print(f"  Base answers loaded: {len(base_records)}")
        print(f"  Compare answers loaded: {len(compare_records)}")

        if not base_records:
            print(f"[WARN] Skip {name}: base ({BASE_TAG}) answers missing at {base_path}.")
            continue
        if not compare_records:
            print(f"[WARN] Skip {name}: compare ({COMPARE_TAG}) answers missing at {compare_path}.")
            continue

        print(f"[INFO] Comparing {name} ({BASE_TAG} vs {COMPARE_TAG})...")
        try:
            eval_records = compute_comparison_records(base_records, compare_records)
            print(f"  Comparison records created: {len(eval_records)}")
            
            if not eval_records:
                print(f"[ERROR] No comparison records generated for {name}")
                continue
            
            metrics = accuracy_from_records(eval_records)
            print(f"  Metrics: {metrics}")
            
            # Write overall dataset row
            row = [name, metrics["total"]]
            row.extend([f"{metrics[m]:.3f}" for m in ["token_f1", "rouge_l", "semantic_similarity"]])
            print(f"  Writing row: {row}")
            writer.writerow(row)
            
            # Write per-category rows
            cat_metrics = metrics_by_category(eval_records)
            for sa_type in sorted(cat_metrics.keys()):
                cat_m = cat_metrics[sa_type]
                cat_row = [f"  {name}:{sa_type}", cat_m["total"]]
                cat_row.extend([f"{cat_m[m]:.3f}" for m in ["token_f1", "rouge_l", "semantic_similarity"]])
                writer.writerow(cat_row)

            all_records.extend(eval_records)
        except Exception as e:
            print(f"[ERROR] {name}: {e}")
            import traceback
            traceback.print_exc()
            continue

    if all_records:
        print(f"[DEBUG] Total records across all datasets: {len(all_records)}")
        metrics = accuracy_from_records(all_records)
        row = ["Combined", metrics["total"]]
        row.extend([f"{metrics[m]:.3f}" for m in ["token_f1", "rouge_l", "semantic_similarity"]])
        writer.writerow(row)
        
        # Write combined per-category rows
        cat_metrics = metrics_by_category(all_records)
        for sa_type in sorted(cat_metrics.keys()):
            cat_m = cat_metrics[sa_type]
            cat_row = [f"  Combined:{sa_type}", cat_m["total"]]
            cat_row.extend([f"{cat_m[m]:.3f}" for m in ["token_f1", "rouge_l", "semantic_similarity"]])
            writer.writerow(cat_row)
    else:
        print(f"[WARN] No all_records to write combined metrics")

print(f"[INFO] Comparison results saved to {out_csv}")

[DEBUG] Current working directory: c:\Users\Dub\Documents\GitHub\CAN_QA\QA\Create_QA_ShortAnswer_Dataset
[DEBUG] Base model: deepseek-r1:14b (deepseek_r1_14b)
[DEBUG] Compare model: llama3.1:8b (llama3_1_8b)
[DEBUG] Output CSV: LLM_SA_Comparison\Comparison_deepseek_r1_14b_vs_llama3_1_8b.csv
[DEBUG] Processing DoS:
  Base answers path: DoS_sa_qa\llm_answers\dos_sa_answers_deepseek_r1_14b.jsonl -> exists: True
  Compare answers path: DoS_sa_qa\llm_answers\dos_sa_answers_llama3_1_8b.jsonl -> exists: True
  Base answers loaded: 916
  Compare answers loaded: 916
[INFO] Comparing DoS (deepseek_r1_14b vs llama3_1_8b)...
  Comparison records created: 916
  Metrics: {'exact_match': 0.0, 'token_f1': 0.0, 'rouge_l': 0.0, 'total': 916}
  Writing row: ['DoS', 916, '0.000', '0.000', '0.000']
[DEBUG] Processing Fuzzy:
  Base answers path: Fuzzy_sa_qa\llm_answers\fuzzy_sa_answers_deepseek_r1_14b.jsonl -> exists: True
  Compare answers path: Fuzzy_sa_qa\llm_answers\fuzzy_sa_answers_llama3_1_8b.jsonl ->